### Déterminer l'échelle d'aggrégation
Ce notebook détermine le nombre d'utilisateurs, de messages, et de messages politiques restant en fonction de l'échelle temporelle d'agrégation et du nombre minimal de message par utilisateur sur cette période pour apparaitre dans le panel.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def stats_panel(df,freq="W",min_obs=5,indiv_col="user",date_col="date",special_col="user_political"):
    data = df.copy()

    # Création de la période temporelle
    data["period"] = data[date_col].dt.to_period(freq)
    counts = (
        data
        .groupby([indiv_col, "period"])
        .size()
        .reset_index(name="n_obs")
    )
    valid_periods = counts[counts["n_obs"] >= min_obs]
    filtered = data.merge(
        valid_periods[[indiv_col, "period"]],
        on=[indiv_col, "period"],
        how="inner"
    )

    n_indiv = filtered[indiv_col].nunique()
    n_points = len(filtered)
    n_periods = (
        filtered[[indiv_col, "period"]]
        .drop_duplicates()
        .shape[0]
    )

    n_special = filtered[special_col].sum()
    special_share = filtered[special_col].mean()

    return {
        "freq": freq,
        "min_obs": min_obs,
        "n_indiv": n_indiv,
        "n_points": n_points,
        "n_indiv_periods": n_periods,
        "n_special": int(n_special),
        "special_share": special_share
    }

Importation des interactions structurées classifiées

In [ ]:
df = pd.read_csv('../clean_data/struct_classified_interactions.csv')
df['date'] = pd.to_datetime(df["date"])

Calcul des observations restantes

In [ ]:
# Fréquences testées
freq_labels = [
    "1D",   # jour
    "3D",   # 3 jours
    "W",   # semaine
    "2W",   # 2 semaines
    "M",   # mois
    "3M"    # trimestre glissant
]

# Seuils minimum d'observations
min_obs_values = [1, 3, 5, 8, 9, 10, 12, 15, 20]

heatmap1 = np.zeros((len(min_obs_values), len(freq_labels)))
heatmap2 = np.zeros((len(min_obs_values), len(freq_labels)))
heatmap3 = np.zeros((len(min_obs_values), len(freq_labels)))

for i, min_obs in enumerate(min_obs_values):
    for j, freq_label in enumerate(freq_labels):

        x = stats_panel(df, freq=freq_label, min_obs=min_obs)
        n_points = x['n_points']
        n_points_polt = x['n_special']
        n_users = x['n_indiv']

        heatmap1[i, j] = n_points
        heatmap2[i, j] = n_points_polt
        heatmap3[i, j] = n_users

Affichage des résultats

In [ ]:
for heatmap in [heatmap1, heatmap2, heatmap3]:
    fig, ax = plt.subplots(figsize=(10, 6))

    im = ax.imshow(
        heatmap,
        aspect="auto",
        origin="lower"
    )

    # Axes
    ax.set_xticks(range(len(freq_labels)))
    ax.set_xticklabels(freq_labels)

    ax.set_yticks(range(len(min_obs_values)))
    ax.set_yticklabels(min_obs_values)

    ax.set_xlabel("Fréquence temporelle")
    ax.set_ylabel("Minimum d'observations")
    ax.set_title("Nombre de points conservés")

    for i in range(len(min_obs_values)):
        for j in range(len(freq_labels)):
            ax.text(
                j,
                i,
                int(heatmap[i, j]),
                ha="center",
                va="center",
                fontsize=9
            )

    cbar = plt.colorbar(im)
    cbar.set_label("Nombre de points")

    plt.tight_layout()
    plt.show()